In [ ]:
#https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india?select=city_hour.csv

# Air Quality - Previsão da qualidade do ar

## Classificação Multiclasse IoT - Previsão da qualidade do ar

**Dataset: India AQI**

Sensores utilizados:

PM2.5: Material particulado fino (μg/m³)

PM10: Material particulado inalável (μg/m³)

NO: Monóxido de nitrogênio (μg/m³)

NO2: Dióxido de nitrogênio (μg/m³)

NOx: Óxidos de nitrogênio (ppb)

NH3: Amônia (μg/m³)

CO: Monóxido de carbono (mg/m³)

SO2: Dióxido de enxofre (μg/m³)

O3: Ozônio (μg/m³)

Benzene: Benzeno (μg/m³)

Toluene: Tolueno (μg/m³)

Xylene: Xileno (μg/m³)

AQI - Índice de qualidade do ar (contínuo)

AQI_Bucket - Categoria do índice de qualidade do ar (classe - 6 classes: Bom, Satisfatório, Moderado, Ruim, Muito Ruim, Perigoso)

## **Importações de bibliotecas**

In [1]:
# Versoes fixas: o modelo .keras e o preprocess .pkl gerados aqui sao carregados
# pela API com EXATAMENTE estas versoes (api/requirements.txt). As duas listas
# tem de continuar iguais -- e por isso que elas sao pinadas dos dois lados.
#
# tensorflow 2.20, numpy 2.1 e pandas 2.2.3 sao os primeiros com wheel para o
# Python 3.13 do Colab. As versoes anteriores nao instalam mais la.
!pip install -q tensorflow==2.20.0 keras==3.10.0 scikit-learn==1.5.2 joblib==1.4.2 pandas==2.2.3 numpy==2.1.0 matplotlib==3.9.2 kagglehub

ERROR: Could not find a version that satisfies the requirement tensorflow==2.19.0 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.19.0


In [2]:
import os, shutil, joblib, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

print("tensorflow:", tf.__version__)
print("keras:", tf.keras.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

tensorflow: 2.20.0
keras: 3.13.2
scikit-learn: 1.6.1
joblib: 1.6.0
pandas: 2.2.3
numpy: 2.1.3


## **Carregamento do dataset**

In [3]:
# Minimal: baixar com kagglehub e trazer só o city_hour.csv para o diretório atual
!pip -q install kagglehub

import os, shutil, kagglehub

path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")
src = os.path.join(path, "city_hour.csv")
shutil.copy(src, "./city_hour.csv")
print("Pronto! Arquivo salvo em: ./city_hour.csv")


100%|██████████| 72.9M/72.9M [00:00<00:00, 112MB/s]

Extracting files...


Pronto! Arquivo salvo em: ./city_hour.csv


In [ ]:
df = pd.read_csv("./city_hour.csv")
df.sample(5)

,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
610059,Patna,2019-10-02 04:00:00,38.50,NaN,10.25,33.69,14.95,NaN,0.88,52.84,NaN,NaN,NaN,NaN,105.0,Moderate
307987,Gurugram,2017-02-18 10:00:00,127.98,NaN,1.41,7.65,NaN,NaN,0.61,3.36,8.87,0.02,2.53,NaN,217.0,Poor
292094,Delhi,2020-05-09 03:00:00,57.80,104.23,15.21,19.16,27.15,30.03,0.83,11.42,55.55,1.58,22.03,0.14,129.0,Moderate
261963,Delhi,2016-11-30 16:00:00,174.03,346.00,8.79,67.99,24.49,61.92,0.62,14.11,63.57,5.54,14.92,NaN,438.0,Severe
419719,Jaipur,2019-12-23 16:00:00,18.73,63.30,7.55,26.36,28.12,22.30,0.59,10.14,63.86,0.98,5.78,NaN,107.0,Moderate


In [ ]:
df.describe()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI
count,562787.000000,411138.000000,591243.000000,590753.000000,584651.000000,435333.000000,621358.000000,577502.000000,578667.000000,544229.000000,487268.000000,252046.000000,578795.000000
mean,67.622994,119.075804,17.421755,28.885157,32.287565,23.607959,2.183539,14.038307,34.798979,3.087595,8.660927,3.130537,166.413500
std,74.730496,104.224752,32.095211,29.162194,39.756669,28.831900,10.970514,19.305540,29.806379,16.456599,21.741023,7.834832,162.112729
min,0.010000,0.010000,0.010000,0.010000,0.000000,0.010000,0.000000,0.010000,0.010000,0.000000,0.000000,0.000000,8.000000
25%,26.200000,52.380000,3.840000,10.810000,10.660000,8.120000,0.420000,4.880000,13.420000,0.050000,0.370000,0.100000,79.000000
50%,46.420000,91.500000,7.960000,20.320000,20.790000,15.380000,0.800000,8.370000,26.240000,0.860000,2.590000,0.790000,116.000000
75%,79.490000,147.520000,16.150000,36.350000,37.150000,29.230000,1.370000,14.780000,47.620000,2.750000,8.410000,3.120000,208.000000
max,999.990000,1000.000000,499.990000,499.510000,498.610000,499.970000,498.570000,199.960000,497.620000,498.070000,499.400000,499.990000,3133.000000


## **Pré-processamento**

### *Feature selection*

In [ ]:
# Renomear coluna
df.rename(columns={'PM2.5': 'PM2_5'}, inplace=True)

feature_columns = ["PM2_5","PM10","NO","NO2","NOx","NH3","CO","SO2","O3","Benzene","Toluene","Xylene"]

### *Remover valores ausentes (NaN)*

In [ ]:
df = df.dropna(subset=feature_columns + ["AQI_Bucket"]).copy()

### *Definir target*

O dataset traz o `AQI_Bucket` em **seis** faixas (`Good`, `Satisfactory`,
`Moderate`, `Poor`, `Very Poor`, `Severe`), que é a escala oficial indiana. Para a
operação da fábrica, seis faixas são detalhe demais: o que muda o comportamento
de quem está lá dentro são **três** situações.

| Faixas do dataset | Classe | O que se faz |
|---|---|---|
| Good, Satisfactory, Moderate | `Aceitável` | nada |
| Poor, Very Poor | `Ruim` | atenção, reduzir exposição |
| Severe | `Perigoso` | evacuar |

`Severe` fica sozinha de propósito: é a classe que dispara o alerta, e juntá-la
com `Very Poor` faria o alarme tocar em situação que não exige evacuação.

Ela é rara — por isso o treino usa `class_weight="balanced"` mais adiante.

> **Estes três nomes são um contrato.** O fluxo do n8n compara a resposta do
> modelo com a palavra `Perigoso` para decidir se manda o alerta. Mudar um nome
> aqui não dá erro em lugar nenhum: o ramo do alerta simplesmente para de
> disparar, em silêncio.

In [ ]:
# Mapear as 6 faixas do dataset em Aceitavel, Ruim e Perigoso.
# Os nomes sao contrato com o fluxo do n8n: ele compara com "Perigoso".
OUTCOME_MAP = {
    "Good":         "Aceitável",
    "Satisfactory": "Aceitável",
    "Moderate":     "Aceitável",
    "Poor":         "Ruim",
    "Very Poor":    "Ruim",
    "Severe":       "Perigoso",
}
df["Outcome3"] = df["AQI_Bucket"].map(OUTCOME_MAP)
df = df.dropna(subset=["Outcome3"])

X = df[feature_columns].astype(float)
y = df["Outcome3"].astype(str)

# Confere que as tres classes sobreviveram ao dropna. Se alguma sumir, o
# modelo nasce sem ela e o alerta nunca dispara.
print(df["Outcome3"].value_counts().to_string())
assert set(df["Outcome3"]) == {"Aceitável", "Ruim", "Perigoso"}, \
    "Faltou uma das tres classes: confira o OUTCOME_MAP e o dropna."

### *Encoding da variável target*

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
classes = le.classes_.tolist()
num_classes = len(classes)
print("Classes:", classes)

## **Dados de treinamento**

### *Split treino-teste-validação*

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

### *Feature Scaling*

In [ ]:
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_val_std   = scaler.transform(X_val)
X_test_std  = scaler.transform(X_test)

input_dim = X_train_std.shape[1]
print(f"Tamanhos -> Treino: {len(X_train)} | Val: {len(X_val)} | Teste: {len(X_test)}")


## **Modelagem**

### Lidar com desbalanceamento de classes

In [ ]:
classes_unique = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes_unique, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes_unique, weights)}
print("class_weight:", class_weight)

### *Modelo - Rede Neural (keras)*

In [ ]:
# 1 hidden layer
def build_model(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model(input_dim, num_classes)
model.summary()

## **Treinamento**

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=8, restore_best_weights=True
)

history = model.fit(
    X_train_std, y_train,
    validation_data=(X_val_std, y_val),
    epochs=50,
    batch_size=256,
    callbacks=[early],
    verbose=1
)

#### ***Curvas de treinamento***

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history["accuracy"], label="treino")
plt.plot(history.history["val_accuracy"], label="val")
plt.xlabel("Épocas"); plt.ylabel("Acurácia"); plt.title("Treino vs Validação")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## **Avaliação**

In [ ]:
y_proba = model.predict(X_test_std, verbose=0)
y_pred  = np.argmax(y_proba, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f"\nAcurácia (teste): {acc:.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=classes, digits=4))

import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Previsto"); plt.ylabel("Real")
plt.title("Matriz de Confusão - AQI_Bucket (Teste)")
plt.show()

## **Salvamento do modelo .pkl**

In [ ]:
MODEL_PATH = "modelo_aqi_nn.keras"
PREP_PATH  = "preprocess_aqi.pkl"

# salva o modelo no formato moderno (recomendado)
model.save(MODEL_PATH)

# salva o pipeline de pré-processamento
prep = {
    "feature_columns": feature_columns,
    "scaler": scaler,
    "label_encoder": le,
    "classes": classes
}
joblib.dump(prep, PREP_PATH)

print("\nArquivos salvos:")
print(" ->", MODEL_PATH)
print(" ->", PREP_PATH)


## **Double-check**

In [ ]:
from tensorflow import keras
import numpy as np

modelo_carregado = keras.models.load_model(MODEL_PATH)
prep_carregado = joblib.load(PREP_PATH)

amostra_X = X_test.iloc[[17490]].copy()
real_idx  = int(y_test[17490])
real_lbl  = prep_carregado["classes"][real_idx]

amostra_std = prep_carregado["scaler"].transform(amostra_X).astype(np.float32)

# Preparar com shape fixo (1, n_feats) para evitar retracing
n_feats = amostra_std.shape[1]
_ = modelo_carregado(np.zeros((1, n_feats), dtype=np.float32), training=False)

# Predição (usando a chamada direta do modelo, que já está preparado)
proba = modelo_carregado(amostra_std, training=False).numpy()[0]
pred_idx = int(np.argmax(proba))
pred_lbl = prep_carregado["classes"][pred_idx]

print("\nVerificação:")
print(amostra_X)
print(f"\nReal: {real_lbl} | Previsto: {pred_lbl}")
print("\nProbabilidades:")
print({cls: float(p) for cls, p in zip(prep_carregado['classes'], np.round(proba,4))})

## Visualiza rede neural

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense

G = nx.DiGraph()

# Só entrada + camadas Dense
layers = [model.input_shape[1]]
layer_names = ["INPUT"]

for layer in model.layers:
    if isinstance(layer, Dense):
        layers.append(layer.units)
        if len(layer_names) < 3:
            layer_names.append(f"HIDDEN {len(layer_names)}")
        else:
            layer_names.append("OUTPUT")

pos = {}
node_id = 0
layer_nodes = []

for i, n_nodes in enumerate(layers):
    nodes = []
    for j in range(n_nodes):
        G.add_node(node_id)
        pos[node_id] = (i * 2.5, -j)
        nodes.append(node_id)
        node_id += 1
    layer_nodes.append(nodes)

# conexões
for i in range(len(layer_nodes) - 1):
    for u in layer_nodes[i]:
        for v in layer_nodes[i + 1]:
            G.add_edge(u, v)

# labels
labels = {}

# nomes das features só na entrada
for i, node in enumerate(layer_nodes[0]):
    labels[node] = feature_columns[i] if i < len(feature_columns) else ""

# saída: mostrar classe 0 e classe 1
for i, node in enumerate(layer_nodes[-1]):
    labels[node] = classes[i]

# ocultas sem texto
for layer in layer_nodes[1:-1]:
    for node in layer:
        labels[node] = ""

# cores
color_map = []
for i, layer in enumerate(layer_nodes):
    if i == 0:
        color = "#9ecae1"   # input
    elif i == len(layer_nodes) - 1:
        color = "#f4a261"   # output
    else:
        color = "#8fd18f"   # hidden
    color_map += [color] * len(layer)

plt.figure(figsize=(15, 8))

nx.draw(
    G,
    pos,
    node_size=700,
    node_color=color_map,
    edge_color="gray",
    labels=labels,
    font_size=10
)

# títulos das camadas
for i, name in enumerate(layer_names):
    plt.text(i * 2.5, 0.5, name, ha="center", fontsize=13, fontweight="bold")

# anotação dos dropouts
plt.text(2.5, -10.5, "Após HIDDEN 1: Dropout(0.2)", ha="center", fontsize=11, color="darkred")
plt.text(5.0, -10.5, "Após HIDDEN 2: Dropout(0.1)", ha="center", fontsize=11, color="darkred")

plt.title("Aplicação IoT - AQI - Arquitetura da NN", fontsize=16)
plt.axis("off")
plt.show()